In [1]:
import wmfdata as wmf
import pandas as pd
import numpy as np
from wmfdata import spark,hive
import plotly.express as px

In [ ]:
# Query

regional_uniques = wmf.spark.run("""
SELECT 
    date_format(CONCAT(ud.year, '-', LPAD(ud.month, 2, '0'), '-01'), 'yyyy-MM') AS month,
    ud.country, ud.country_code as country_code,
    SUM(ud.uniques_estimate) AS total_unique_devices,
    cmd.wmf_region
FROM 
    wmf.unique_devices_per_project_family_monthly ud
JOIN 
    gdi.country_meta_data cmd
ON 
    ud.country_code = cmd.country_code_iso_2
WHERE 
    ud.project_family = 'wikipedia'
    AND ud.year in (2023,2022)
    AND ud.month = 11
GROUP BY 
    date_format(CONCAT(ud.year, '-', LPAD(ud.month, 2, '0'), '-01'), 'yyyy-MM'), 
    ud.country, ud.country_code, cmd.wmf_region, ud.country_code




""")
# Convert query results into dataframe
df = regional_uniques.copy()

df['month'] = pd.to_datetime(df['month'], format='%Y-%m')

# Convert the unique months to a Series and sort
unique_months = pd.Series(df['month'].dt.to_period('M').unique()).sort_values(ascending=False)

# Initialize an empty DataFrame to hold all results
final_df = pd.DataFrame()

for current_month in unique_months:
    previous_year_month = current_month.to_timestamp() - pd.DateOffset(years=1)

    # Filter the data for the current month and the same month in the previous year
    filtered_df = df[df['month'].isin([current_month.to_timestamp(), previous_year_month])]

    # Pivot table for the filtered data
    df_pivot = filtered_df.pivot_table(index=['country_code', 'wmf_region'], 
                                       columns='month', 
                                       values='total_unique_devices',
                                       aggfunc='sum').reset_index()
    #Merge
    country_code_name_mapping = df[['country_code', 'country']].drop_duplicates(subset='country_code')

    # Merge the pivot table with the country name from the country_code_name_mapping DataFrame
    df_pivot = pd.merge(df_pivot, country_code_name_mapping, on='country_code', how='left')
    
    # Storing Initial Values
    df_pivot['current_metric'] = df_pivot[current_month.to_timestamp()]
    df_pivot['previous_metric'] = df_pivot.get(previous_year_month, 0)


    # Calculating Absolute Change
    df_pivot['absolute_change'] = df_pivot['current_metric'] - df_pivot['previous_metric']

    # Adding Current and Previous Metrics
    df_pivot['current_metric'] = df_pivot[current_month.to_timestamp()]
    df_pivot['previous_metric'] = df_pivot.get(previous_year_month, 0)

    # Calculate Regional and Global Totals
    regional_totals = df_pivot.groupby('wmf_region')[['current_metric', 'previous_metric']].sum().reset_index()
    global_current_metric_total = df_pivot['current_metric'].sum()
    global_previous_metric_total = df_pivot['previous_metric'].sum()

    # Merge with Regional Totals and add Global Totals
    df_merged = pd.merge(df_pivot, regional_totals, on='wmf_region', suffixes=('', '_regional_total'))
    df_merged['global_current_metric_total'] = global_current_metric_total
    df_merged['global_previous_metric_total'] = global_previous_metric_total

    # Calculating Proportions and Formula Results
    df_merged['proportion_current_regional'] = df_merged['current_metric'] / df_merged['current_metric_regional_total']
    df_merged['proportion_previous_regional'] = df_merged['previous_metric'] / df_merged['previous_metric_regional_total']
    df_merged['proportion_current_global'] = df_merged['current_metric'] / df_merged['global_current_metric_total']
    df_merged['proportion_previous_global'] = df_merged['previous_metric'] / df_merged['global_previous_metric_total']

    df_merged['formula_result_regional'] = abs(((df_merged['current_metric'] - df_merged['previous_metric']) * df_merged['proportion_current_regional']) +
                                               ((df_merged['proportion_current_regional'] - df_merged['proportion_previous_regional']) * (df_merged['previous_metric'] - df_merged['previous_metric_regional_total'])))
    df_merged['formula_result_global'] = abs(((df_merged['current_metric'] - df_merged['previous_metric']) * df_merged['proportion_current_global']) +
                                             ((df_merged['proportion_current_global'] - df_merged['proportion_previous_global']) * (df_merged['previous_metric'] - df_merged['global_previous_metric_total'])))

    # Calculate the Total Sum of 'formula_result' per Region and Globally
    total_formula_result_regional = df_merged.groupby('wmf_region')['formula_result_regional'].sum().reset_index().rename(columns={'formula_result_regional': 'formula_result_regional_sum'})
    total_formula_result_global = df_merged['formula_result_global'].sum()

    # Merge with total formula results and calculate percentages
    df_merged = pd.merge(df_merged, total_formula_result_regional, on='wmf_region')
    df_merged['formula_result_percentage_regional'] = (df_merged['formula_result_regional'] / df_merged['formula_result_regional_sum']) * 100
    df_merged['formula_result_percentage_global'] = (df_merged['formula_result_global'] / total_formula_result_global) * 100

    # Simple Calculation Columns
    total_abs_change_regional = df_merged.groupby('wmf_region')['absolute_change'].apply(lambda x: x.abs().sum()).reset_index().rename(columns={'absolute_change': 'total_abs_change_regional'})
    total_abs_change_global = df_merged['absolute_change'].abs().sum()

    # Merge the total absolute changes with the main DataFrame
    df_merged = pd.merge(df_merged, total_abs_change_regional, on='wmf_region')

    # Calculate the simple calculation columns as percentages
    df_merged['simple_calculation_regional'] = (df_merged['absolute_change'].abs() / df_merged['total_abs_change_regional']) * 100
    df_merged['simple_calculation_global'] = (df_merged['absolute_change'].abs() / total_abs_change_global) * 100

    # Add a 'month' column to df_merged
    df_merged['month'] = current_month.strftime('%Y-%m')
    
    # Add ranking logic
    df_merged['rank_simple_calculation'] = df_merged.groupby(['month', 'wmf_region'])['simple_calculation_regional'].rank(method='dense', ascending=False)
    df_merged['rank_formula_result'] = df_merged.groupby(['month', 'wmf_region'])['formula_result_percentage_regional'].rank(method='dense', ascending=False)


    # Append the results for this month to the final DataFrame
    final_df = pd.concat([final_df, df_merged])


# Convert 'absolute_change' to its absolute value
final_df['absolute_change'] = final_df['absolute_change'].abs()


# Sort the DataFrame by 'month', 'wmf_region', and 'absolute_change'
final_df = final_df.sort_values(by=['month', 'wmf_region', 'absolute_change'], ascending=[False, True, False])

# Save the sorted DataFrame to a CSV file (2 different files depending on if this is the entire history or just YoY)
if len(final_df.month.unique()) > 2:
    print("Saving CSV file")
    
    # Select required columns
    final_df = final_df[['month', 'country', 'wmf_region', 'current_metric', 'previous_metric', 'absolute_change', 
                         'formula_result_percentage_regional', 'simple_calculation_regional', 
                         'rank_simple_calculation', 'rank_formula_result',
                         'simple_calculation_global', 'formula_result_percentage_global']]
    final_df.to_csv("unique_devices_time_series_analysis.csv", index=False)

else:
    print("Saving CSV file")
    # Select required columns
    final_df = final_df[['month', 'country', 'wmf_region', 'current_metric', 'previous_metric', 'absolute_change', 
                          'simple_calculation_regional', 
                         'simple_calculation_global','formula_result_percentage_regional', 'formula_result_percentage_global']]
    latest_month = final_df['month'].max()
    final_df = final_df[final_df['month'] == latest_month]
    final_df.to_csv("unique_devices_time_series_yoy.csv", index = False)
    
# Show df
final_df

## Mobile enwiki breakdown

In [13]:
# Query

regional_uniques_mobile = wmf.spark.run("""
SELECT 
    date_format(CONCAT(ud.year, '-', LPAD(ud.month, 2, '0'), '-01'), 'yyyy-MM') AS month,
    ud.country, ud.country_code as country_code,
    SUM(ud.uniques_estimate) AS total_unique_devices,
    cmd.wmf_region
FROM 
    wmf.unique_devices_per_domain_monthly ud
JOIN 
    gdi.country_meta_data cmd
ON 
    ud.country_code = cmd.country_code_iso_2
WHERE 
    ud.domain in ('es.m.wikipedia.org', 'pt.m.wikipedia.org')
    AND ud.year in (2023,2022)
    AND ud.month = 11
    AND cmd.wmf_region = 'Latin America & Caribbean'
GROUP BY 
    date_format(CONCAT(ud.year, '-', LPAD(ud.month, 2, '0'), '-01'), 'yyyy-MM'), 
    ud.country, ud.country_code, cmd.wmf_region, ud.country_code




""")

# Convert query results into dataframe
df = regional_uniques_mobile.copy()

df['month'] = pd.to_datetime(df['month'], format='%Y-%m')

# Convert the unique months to a Series and sort
unique_months = pd.Series(df['month'].dt.to_period('M').unique()).sort_values(ascending=False)

final_df = pd.DataFrame()

for current_month in unique_months:
    previous_year_month = current_month.to_timestamp() - pd.DateOffset(years=1)

    # Filter the data for the current month and the same month in the previous year
    filtered_df = df[df['month'].isin([current_month.to_timestamp(), previous_year_month])]

    # Pivot table for the filtered data
    df_pivot = filtered_df.pivot_table(index=['country_code', 'wmf_region'], 
                                       columns='month', 
                                       values='total_unique_devices',
                                       aggfunc='sum').reset_index()
    #Merge
    country_code_name_mapping = df[['country_code', 'country']].drop_duplicates(subset='country_code')

    # Merge the pivot table with the country name from the country_code_name_mapping DataFrame
    df_pivot = pd.merge(df_pivot, country_code_name_mapping, on='country_code', how='left')
    
    # Storing Initial Values
    df_pivot['current_metric'] = df_pivot[current_month.to_timestamp()]
    df_pivot['previous_metric'] = df_pivot.get(previous_year_month, 0)


    # Calculating Absolute Change
    df_pivot['absolute_change'] = df_pivot['current_metric'] - df_pivot['previous_metric']

    # Adding Current and Previous Metrics
    df_pivot['current_metric'] = df_pivot[current_month.to_timestamp()]
    df_pivot['previous_metric'] = df_pivot.get(previous_year_month, 0)

    # Calculate Regional and Global Totals
    regional_totals = df_pivot.groupby('wmf_region')[['current_metric', 'previous_metric']].sum().reset_index()
    global_current_metric_total = df_pivot['current_metric'].sum()
    global_previous_metric_total = df_pivot['previous_metric'].sum()

    # Merge with Regional Totals and add Global Totals
    df_merged = pd.merge(df_pivot, regional_totals, on='wmf_region', suffixes=('', '_regional_total'))
    df_merged['global_current_metric_total'] = global_current_metric_total
    df_merged['global_previous_metric_total'] = global_previous_metric_total

    # Calculating Proportions and Formula Results
    df_merged['proportion_current_regional'] = df_merged['current_metric'] / df_merged['current_metric_regional_total']
    df_merged['proportion_previous_regional'] = df_merged['previous_metric'] / df_merged['previous_metric_regional_total']
    df_merged['proportion_current_global'] = df_merged['current_metric'] / df_merged['global_current_metric_total']
    df_merged['proportion_previous_global'] = df_merged['previous_metric'] / df_merged['global_previous_metric_total']

    df_merged['formula_result_regional'] = abs(((df_merged['current_metric'] - df_merged['previous_metric']) * df_merged['proportion_current_regional']) +
                                               ((df_merged['proportion_current_regional'] - df_merged['proportion_previous_regional']) * (df_merged['previous_metric'] - df_merged['previous_metric_regional_total'])))
    df_merged['formula_result_global'] = abs(((df_merged['current_metric'] - df_merged['previous_metric']) * df_merged['proportion_current_global']) +
                                             ((df_merged['proportion_current_global'] - df_merged['proportion_previous_global']) * (df_merged['previous_metric'] - df_merged['global_previous_metric_total'])))

    # Calculate the Total Sum of 'formula_result' per Region and Globally
    total_formula_result_regional = df_merged.groupby('wmf_region')['formula_result_regional'].sum().reset_index().rename(columns={'formula_result_regional': 'formula_result_regional_sum'})
    total_formula_result_global = df_merged['formula_result_global'].sum()

    # Merge with total formula results and calculate percentages
    df_merged = pd.merge(df_merged, total_formula_result_regional, on='wmf_region')
    df_merged['formula_result_percentage_regional'] = (df_merged['formula_result_regional'] / df_merged['formula_result_regional_sum']) * 100
    df_merged['formula_result_percentage_global'] = (df_merged['formula_result_global'] / total_formula_result_global) * 100

    # Simple Calculation Columns
    total_abs_change_regional = df_merged.groupby('wmf_region')['absolute_change'].apply(lambda x: x.abs().sum()).reset_index().rename(columns={'absolute_change': 'total_abs_change_regional'})
    total_abs_change_global = df_merged['absolute_change'].abs().sum()

    # Merge the total absolute changes with the main DataFrame
    df_merged = pd.merge(df_merged, total_abs_change_regional, on='wmf_region')

    # Calculate the simple calculation columns as percentages
    df_merged['simple_calculation_regional'] = (df_merged['absolute_change'].abs() / df_merged['total_abs_change_regional']) * 100
    df_merged['simple_calculation_global'] = (df_merged['absolute_change'].abs() / total_abs_change_global) * 100

    # Add a 'month' column to df_merged
    df_merged['month'] = current_month.strftime('%Y-%m')
    
    # Add ranking logic
    df_merged['rank_simple_calculation'] = df_merged.groupby(['month', 'wmf_region'])['simple_calculation_regional'].rank(method='dense', ascending=False)
    df_merged['rank_formula_result'] = df_merged.groupby(['month', 'wmf_region'])['formula_result_percentage_regional'].rank(method='dense', ascending=False)


    # Append the results for this month to the final DataFrame
    final_df = pd.concat([final_df, df_merged])


# Convert 'absolute_change' to its absolute value
final_df['absolute_change'] = final_df['absolute_change'].abs()


# Sort the DataFrame by 'month', 'wmf_region', and 'absolute_change'
final_df = final_df.sort_values(by=['month', 'wmf_region', 'absolute_change'], ascending=[False, True, False])

# Save the sorted DataFrame to a CSV file (2 different files depending on if this is the entire history or just YoY)
if len(final_df.month.unique()) > 2:
    print("Saving CSV file")
    
    # Select required columns
    final_df = final_df[['month', 'country', 'wmf_region', 'current_metric', 'previous_metric', 'absolute_change', 
                         'formula_result_percentage_regional', 'simple_calculation_regional', 
                         'rank_simple_calculation', 'rank_formula_result',
                         'simple_calculation_global', 'formula_result_percentage_global']]
    final_df.to_csv("unique_devices_time_series_analysis_mobile.csv", index=False)

else:
    print("Saving CSV file")
    # Select required columns
    final_df = final_df[['month', 'country', 'wmf_region', 'current_metric', 'previous_metric', 'absolute_change', 
                          'simple_calculation_regional', 
                         'simple_calculation_global','formula_result_percentage_regional', 'formula_result_percentage_global']]
    latest_month = final_df['month'].max()
    final_df = final_df[final_df['month'] == latest_month]
    final_df.to_csv("unique_devices_time_series_yoy_mobile.csv", index = False)
    
# Show df
final_df

Saving CSV file


,month,country,wmf_region,current_metric,previous_metric,absolute_change,simple_calculation_regional,simple_calculation_global,formula_result_percentage_regional,formula_result_percentage_global
9,2023-11,Brazil,Latin America & Caribbean,27838881,34414175,6575294,42.429825,42.429825,9.756833,9.756833
35,2023-11,Mexico,Latin America & Caribbean,21351238,23974122,2622884,16.925252,16.925252,30.242770,30.242770
2,2023-11,Argentina,Latin America & Caribbean,10944929,12549913,1604984,10.356828,10.356828,7.519907,7.519907
13,2023-11,Colombia,Latin America & Caribbean,8140218,9573972,1433754,9.251895,9.251895,0.421136,0.421136
38,2023-11,Peru,Latin America & Caribbean,4676162,5371429,695267,4.486500,4.486500,1.923129,1.923129
12,2023-11,Chile,Latin America & Caribbean,5723736,6241954,518218,3.344018,3.344018,10.234793,10.234793
19,2023-11,Ecuador,Latin America & Caribbean,2399696,2913443,513747,3.315167,3.315167,3.088040,3.088040
7,2023-11,Bolivia,Latin America & Caribbean,1526224,1832224,306000,1.974593,1.974593,1.540460,1.540460
18,2023-11,Dominican Republic,Latin America & Caribbean,1843424,2104715,261291,1.686089,1.686089,0.882481,0.882481
14,2023-11,Costa Rica,Latin America & Caribbean,923386,1087175,163789,1.056917,1.056917,0.424981,0.424981
